<a href="https://colab.research.google.com/github/FamsGit/Komputasi-Numerik/blob/main/02-Sistem-Persamaan-Linear/SPL_Eliminasi_Gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Penyelesaian Sistem Persamaan Linear Tiga Variabel

**Metode:** Eliminasi Gauss dengan *partial pivoting* dan substitusi mundur  
**Nama:** Muhammad Fahmi Amrulloh — **NIM:** 202511420045

Notebook ini disusun per cell. Jalankan dari atas ke bawah menggunakan **Runtime → Run all**. Setelah cell fungsi dijalankan, cell pengujian dapat dijalankan kembali dengan mengganti input.

## 1. Dasar masalah

Kita akan menyelesaikan sistem:

\[
\begin{cases}
4x+y-z=3,\\
2x+5y+z=15,\\
x-y+3z=8.
\end{cases}
\]

Dalam bentuk matriks, masalahnya adalah `A x = b`. Eliminasi Gauss mengubah matriks augmented menjadi bentuk segitiga atas. *Partial pivoting* memilih elemen pivot dengan nilai absolut terbesar pada kolom aktif untuk mengurangi risiko pembagian oleh bilangan yang sangat kecil.

## 2. Memasukkan data

Cell berikut berisi koefisien matriks `A` dan vektor konstanta `b`. Setiap baris `A` mewakili koefisien variabel `x`, `y`, dan `z` dari satu persamaan. Elemen pada posisi yang sama di `b` adalah konstanta ruas kanan.

In [ ]:
# Matriks koefisien dari tiga persamaan (kolom: x, y, z).
A = [
    [4.0,  1.0, -1.0],  # Persamaan pertama: 4x + y - z = 3
    [2.0,  5.0,  1.0],  # Persamaan kedua: 2x + 5y + z = 15
    [1.0, -1.0,  3.0],  # Persamaan ketiga: x - y + 3z = 8
]

# Vektor konstanta di sisi kanan setiap persamaan.
b = [3.0, 15.0, 8.0]

# Tampilkan ukuran data untuk memastikan ada tiga persamaan dan tiga variabel.
print(f"Jumlah persamaan: {len(A)}")
print(f"Jumlah variabel : {len(A[0])}")
print("Vektor konstanta:", b)

Jumlah persamaan: 3
Jumlah variabel : 3
Vektor konstanta: [3.0, 15.0, 8.0]


## 3. Membuat fungsi penampil matriks

Fungsi `tampilkan_matriks` hanya membantu memperjelas proses. Parameter `M` adalah matriks yang akan dicetak dan `judul` adalah label tahap. Format `:10.4f` menampilkan angka dengan empat angka desimal dan lebar bidang yang cukup agar kolom mudah dibandingkan.

In [ ]:
def tampilkan_matriks(M, judul):
    # Cetak nama tahap agar pembaca mengetahui matriks yang sedang dilihat.
    print(judul)

    # Telusuri setiap baris matriks satu per satu.
    for baris in M:
        # Format tiap angka menjadi lebar 10 karakter dengan 4 angka desimal.
        isi_baris = " ".join(f"{nilai:10.4f}" for nilai in baris)

        # Tampilkan baris yang sudah dirapikan.
        print(isi_baris)

    # Baris kosong memisahkan keluaran dari tahap berikutnya.
    print()

## 4. Implementasi Eliminasi Gauss dengan partial pivoting

Penjelasan alur fungsi:

1. Salin data `A` dan `b` agar data masukan asli tidak berubah.
2. Periksa bahwa jumlah baris, kolom, dan konstanta konsisten.
3. Bentuk matriks augmented `[A | b]`.
4. Pada setiap kolom pivot, pilih baris dengan nilai absolut pivot terbesar.
5. Tukar baris bila diperlukan, lalu eliminasi seluruh elemen di bawah pivot.
6. Setelah matriks segitiga atas terbentuk, lakukan substitusi mundur.
7. Hitung residual `A x - b` sebagai pemeriksaan hasil.

Kode diberi komentar pada setiap langkah penting agar prosesnya dapat ditelusuri.

In [ ]:
def eliminasi_gauss_partial_pivoting(A, b, toleransi=1e-12):
    # Salin matriks dan vektor sehingga masukan asli tidak dimodifikasi.
    matriks = [baris[:] for baris in A]
    konstanta = b[:]

    # Tentukan banyaknya persamaan atau ukuran sistem.
    n = len(matriks)

    # Pastikan matriks memiliki setidaknya satu baris.
    if n == 0:
        raise ValueError("Matriks A tidak boleh kosong.")

    # Pastikan setiap baris mempunyai tepat n koefisien (matriks persegi).
    if any(len(baris) != n for baris in matriks):
        raise ValueError("Matriks A harus berukuran n x n.")

    # Pastikan jumlah konstanta sama dengan jumlah persamaan.
    if len(konstanta) != n:
        raise ValueError("Panjang vektor b harus sama dengan jumlah baris A.")

    # Gabungkan koefisien dan konstanta menjadi matriks augmented.
    augmented = [matriks[i] + [konstanta[i]] for i in range(n)]

    # Simpan rekaman matriks setelah setiap tahap eliminasi untuk ditampilkan.
    riwayat = [([baris[:] for baris in augmented], "Matriks augmented awal")]

    # Lakukan eliminasi maju untuk setiap kolom pivot.
    for k in range(n):
        # Cari baris dengan nilai absolut terbesar pada kolom k mulai baris k.
        baris_pivot = max(range(k, n), key=lambda i: abs(augmented[i][k]))

        # Hentikan jika pivot terlalu kecil sehingga pembagian tidak aman.
        if abs(augmented[baris_pivot][k]) < toleransi:
            raise ValueError("Pivot mendekati nol; sistem tidak memiliki solusi tunggal.")

        # Tukar posisi baris aktif dengan baris pivot terpilih jika perlu.
        if baris_pivot != k:
            augmented[k], augmented[baris_pivot] = augmented[baris_pivot], augmented[k]

        # Telusuri semua baris yang berada di bawah baris pivot.
        for i in range(k + 1, n):
            # Hitung faktor pengali untuk menghilangkan elemen pada kolom pivot.
            faktor = augmented[i][k] / augmented[k][k]

            # Elemen di bawah pivot ditetapkan nol secara eksplisit.
            augmented[i][k] = 0.0

            # Perbarui seluruh elemen di sisi kanan pivot, termasuk konstanta.
            for j in range(k + 1, n + 1):
                augmented[i][j] -= faktor * augmented[k][j]

        # Simpan salinan matriks setelah kolom pivot k selesai dieliminasi.
        riwayat.append(([baris[:] for baris in augmented], f"Setelah eliminasi kolom {k + 1}"))

    # Siapkan list kosong untuk menampung nilai variabel x, y, dan z.
    solusi = [0.0] * n

    # Hitung variabel dari baris terakhir menuju baris pertama (substitusi mundur).
    for i in range(n - 1, -1, -1):
        # Hitung kontribusi variabel yang sudah diketahui di sebelah kanan diagonal.
        jumlah_kanan = sum(augmented[i][j] * solusi[j] for j in range(i + 1, n))

        # Periksa kembali diagonal sebelum membagi.
        if abs(augmented[i][i]) < toleransi:
            raise ValueError("Elemen diagonal mendekati nol saat substitusi mundur.")

        # Isolasi variabel saat ini menggunakan persamaan pada baris ke-i.
        solusi[i] = (augmented[i][n] - jumlah_kanan) / augmented[i][i]

    # Hitung residual asli A*x - b untuk memeriksa ketepatan solusi.
    residual = [
        sum(A[i][j] * solusi[j] for j in range(n)) - b[i]
        for i in range(n)
    ]

    # Kembalikan solusi, matriks akhir, riwayat proses, dan residual.
    return solusi, augmented, riwayat, residual

## 5. Menjalankan fungsi dan melihat proses eliminasi

Panggil fungsi dengan `A` dan `b`. Hasilnya terdiri dari empat bagian: vektor solusi, matriks augmented akhir, riwayat matriks tiap tahap, dan residual. Loop berikut menampilkan riwayat agar perubahan matriks terlihat.

In [ ]:
# Jalankan metode Eliminasi Gauss pada data contoh.
solusi, matriks_akhir, riwayat, residual = eliminasi_gauss_partial_pivoting(A, b)

# Tampilkan matriks augmented awal dan hasil dari setiap tahap eliminasi.
for matriks_tahap, nama_tahap in riwayat:
    tampilkan_matriks(matriks_tahap, nama_tahap)

# Tampilkan matriks segitiga atas yang menjadi dasar substitusi mundur.
tampilkan_matriks(matriks_akhir, "Matriks segitiga atas akhir")

Matriks augmented awal
    4.0000     1.0000    -1.0000     3.0000
    2.0000     5.0000     1.0000    15.0000
    1.0000    -1.0000     3.0000     8.0000

Setelah eliminasi kolom 1
    4.0000     1.0000    -1.0000     3.0000
    0.0000     4.5000     1.5000    13.5000
    0.0000    -1.2500     3.2500     7.2500

Setelah eliminasi kolom 2
    4.0000     1.0000    -1.0000     3.0000
    0.0000     4.5000     1.5000    13.5000
    0.0000     0.0000     3.6667    11.0000

Setelah eliminasi kolom 3
    4.0000     1.0000    -1.0000     3.0000
    0.0000     4.5000     1.5000    13.5000
    0.0000     0.0000     3.6667    11.0000

Matriks segitiga atas akhir
    4.0000     1.0000    -1.0000     3.0000
    0.0000     4.5000     1.5000    13.5000
    0.0000     0.0000     3.6667    11.0000



## 6. Menampilkan solusi dan pemeriksaan residual

Urutan solusi mengikuti urutan kolom matriks, yaitu `x`, `y`, dan `z`. Residual adalah selisih antara ruas kiri hasil substitusi dan ruas kanan asli. Nilai residual yang mendekati nol menunjukkan bahwa solusi memenuhi persamaan.

In [ ]:
# Pasangkan setiap nilai solusi dengan nama variabelnya.
nama_variabel = ["x", "y", "z"]

# Tampilkan nilai solusi dengan enam angka di belakang koma.
for nama, nilai in zip(nama_variabel, solusi):
    print(f"{nama} = {nilai:.6f}")

# Tampilkan residual setiap persamaan untuk menguji hasil perhitungan.
print("\nResidual A*x - b:")
for nomor, nilai in enumerate(residual, start=1):
    print(f"Persamaan {nomor}: {nilai:.12f}")

# Hitung residual maksimum sebagai ringkasan tingkat kesalahan numerik.
residual_maksimum = max(abs(nilai) for nilai in residual)
print(f"\nResidual maksimum: {residual_maksimum:.12e}")

x = 1.000000
y = 2.000000
z = 3.000000

Residual A*x - b:
Persamaan 1: 0.000000000000
Persamaan 2: 0.000000000000
Persamaan 3: 0.000000000000

Residual maksimum: 0.000000000000e+00


## 7. Kesimpulan

Untuk sistem persamaan contoh, hasil yang diharapkan adalah `x = 1`, `y = 2`, dan `z = 3`. Program menggabungkan eliminasi maju, pemilihan pivot parsial, substitusi mundur, dan pemeriksaan residual. Coba ubah koefisien pada cell input untuk menguji sistem lain yang memiliki solusi tunggal.